# Una temperatura efectiva amb terme radiatiu — calibratge i validació

**Objectiu.** Construir una temperatura efectiva $T_\mathrm{rad}$ que afegeixi el canal de radiació
(sol) a la teva temperatura corregida actual `<T-T_fixed+<T>>`, i comprovar **honestament** si separa
els estats de confort millor que la temperatura corregida i que l'Humidex.

$$T_\mathrm{env} = T_\mathrm{corr} + k_1\, f_\mathrm{sun} + k_2\, f_\mathrm{wind}$$

on $f_\mathrm{mix}, f_\mathrm{sun}$ són les **fraccions** de vots "barreja de sol i ombra" i "ple sol"
a cada parada (referència = ombra plena). Els coeficients $k_1, k_2$ **no es fixen a priori**: es
calibren amb les dades.

**Pla d'honestedat** (el que separa això d'un exercici de *physics envy*):
1. Calibratge per maximitzar la separació confort–temperatura (Spearman $|\rho|$ a nivell de vot).
2. Comprovació física: el $k_2$ òptim hauria de caure al rang de la literatura de radiació (~10–15 °C per a sol ple).
3. Incertesa amb **bootstrap agrupat per caminada** (no per vot): si la millora no exclou el zero, el sol no ajuda de manera robusta.
4. **Control de règim**: la millora, ¿és *dins* de caminada o només *entre* caminades? Si és només entre caminades, el sol és una variable de **forçament/règim**, no una coordenada local — i això és un resultat, no un fracàs.


In [ ]:
import pandas as pd, numpy as np, warnings
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
np.random.seed(0)
plt.rcParams.update({'figure.dpi':110,'font.size':10})

In [ ]:
# ----------------------- CONFIG -----------------------
CSV = '../../data/all_surveys(stops).csv'   # ajusta el path si cal

#7 categories de confort -> puntuacio ordinal (objectiu de separacio)
SCORE = {'Very uncomfortable':-3,'Uncomfortable':-2,'Slightly uncomfortable':-1,
         'Neutral(comfort)':0,'Slightly comfortable':1,'Comfortable':2,'Very comfortable':3}

SCORE3 = {'Very uncomfortable':-3,'Uncomfortable':-2,'Slightly uncomfortable':-1,
          'Neutral(comfort)':0,'Slightly comfortable':1,'Comfortable':2,'Very comfortable':3}

COMF = list(SCORE.keys())

# agrupacio de 3 estats (nomes per a la narrativa Kruskal-Wallis; el calibratge usa l'ordinal)
STATE3 = {'Very uncomfortable':'unc','Uncomfortable':'unc','Slightly uncomfortable':'unc',
          'Neutral(comfort)':'neu','Slightly comfortable':'comf',
          'Comfortable':'comf','Very comfortable':'comf'}

# ----------------------- CONFIG -----------------------
 # ajusta el path si cal

# 7 categories de sensació tèrmica -> puntuació ordinal
# Nota: com que no tens "Cold", fem una escala ordinal monotònica.
# El valor exacte importa menys que l'ordre si fas separabilitat / Kruskal / regressió ordinal simple.
# SENS_SCORE = {
#     'Cool': -2,
#     'Slightly cool': -1,
#     'Neutral(sensation)': 0,
#     'Slightly warm': 1,
#     'Warm': 2,
#     'Hot': 3,
#     'Very hot': 4
# }

# SENS3_SCORE = {
#     'Cool': -1,
#     'Slightly cool': -1,
#     'Neutral(sensation)': 0,
#     'Slightly warm': 1,
#     'Warm': 1,
#     'Hot': 1,
#     'Very hot': 1
# }

# SENS = list(SENS3_SCORE.keys())

# # Agrupació en 3 estats: cool / neutral / hot
# STATE3_SENS = {
#     'Cool': 'cool',
#     'Slightly cool': 'cool',
#     'Neutral(sensation)': 'neutral',
#     'Slightly warm': 'hot',
#     'Warm': 'hot',
#     'Hot': 'hot',
#     'Very hot': 'hot'
# }



WIND = [
    "It's not windy",
    "A very light wind",
    "A light wind",
    "A moderate wind",
    "A strong wind"
]


SUN = ['In full sun','In a mixture of sun and shadow','In full shade']
T_COL   = '<T-T_fixed+<T>>'      # la teva temperatura corregida actual
HDX_COL = '<HDX-HDX_fixed+<HDX>>'

In [ ]:
# ----------------------- LOAD + CLEAN -----------------------
df = pd.read_csv(CSV)
# normalitza el typo de ciutat ('Llobreat' -> 'Llobregat') -> 5 ciutats
df['city'] = df['city'].str.replace('Llobreat','Llobregat', regex=False).str.strip()
# identificador de caminada: data + driver + ciutat  (=> 48 caminades)
df['walk'] = df['date'].astype(str)+'|'+df['driver_name'].astype(str)+'|'+df['city'].astype(str)

# --- MODEL FINAL: T_env = T_corr + k_sun*f_sun - k_wind*wind_sc ---
# Afegir wind_sc als features (cel·la de features)
WIND = ["It's not windy","A very light wind","A light wind","A moderate wind","A strong wind"]
df[WIND] = df[WIND].fillna(0)
wind_tot = df[WIND].sum(axis=1).replace(0, np.nan)
df['wind_sc'] = (df[WIND] * np.array([0,1,2,3,4])).sum(axis=1) / (wind_tot * 4)
# i afegir 'wind_sc' a les columnes de l'expansió de vots

df[COMF] = df[COMF].fillna(0)
df[SUN]  = df[SUN].fillna(0)
sun_tot  = df[SUN].sum(axis=1).replace(0, np.nan)
df['f_sun'] = df['In full sun']/sun_tot
df['f_mix'] = df['In a mixture of sun and shadow']/sun_tot
df['T_corr']   = df[T_COL]
df['HDX_corr'] = df[HDX_COL]
df['SVF_n']    = pd.to_numeric(df['SVF'], errors='coerce')

df = df.dropna(subset=['T_corr','f_sun']).reset_index(drop=True)
print(f"parades: {len(df)} | caminades: {df['walk'].nunique()} | ciutats: {df['city'].nunique()}")
print(f"T_corr  rang: {df.T_corr.min():.1f}-{df.T_corr.max():.1f} C  (mitjana {df.T_corr.mean():.1f})")

In [ ]:
# ----------------------- EXPAND TO VOTE LEVEL -----------------------
# cada vot hereta les covariables (a nivell de parada) de la seva parada
rows = []
for _, r in df.iterrows():
    for c in COMF:
        n = int(round(r[c]))
        if n > 0:
            rows += [(r['walk'], SCORE3[c], STATE3[c], r['T_corr'], r['HDX_corr'],
                      r['f_sun'], r['f_mix'], r['SVF_n'], r['wind_sc'])]*n
v = pd.DataFrame(rows, columns=['walk','score','state3','T_corr','HDX_corr','f_sun','f_mix','SVF_n','wind_sc'])
v['disc'] = (v['score'] < 0).astype(int)     # descomfort = pitjor que neutre
print('vots:', len(v))
print(v['state3'].value_counts().to_dict())

In [ ]:
# ----------------------- METRIQUES DE SEPARACIO -----------------------
def Trad(d, k1, k2):
    return d['T_corr'] + k1*d['f_mix'] + k2*d['f_sun']

def abs_rho(temp, score):
    "Spearman |rho| temperatura vs puntuacio de confort (a nivell de vot)"
    return abs(stats.spearmanr(temp, score).statistic)

def cv_auc(col, frame=None):
    "AUC del descomfort amb GroupKFold per caminada (control de fuita participant/caminada)"
    d = v if frame is None else frame
    X = d[[col]].values; y = d['disc'].values; g = d['walk'].values
    aucs = []
    for tr, te in GroupKFold(5).split(X, y, g):
        m = LogisticRegression().fit(X[tr], y[tr])
        aucs.append(roc_auc_score(y[te], m.predict_proba(X[te])[:,1]))
    return float(np.mean(aucs))

def kw_H(col):
    "Kruskal-Wallis H entre els 3 estats"
    groups = [g[col].values for _, g in v.groupby('state3')]
    return stats.kruskal(*groups).statistic

In [ ]:
# ----------------------- BASELINE -----------------------
print('--- separacio basal (sense terme de sol) ---')
print(f"|rho|  T_corr  = {abs_rho(v.T_corr, v.score):.3f}")
print(f"|rho|  HDX     = {abs_rho(v.HDX_corr, v.score):.3f}")
print(f"|rho|  f_sun   = {abs_rho(v.f_sun,  v.score):.3f}   (el sol sol, com a referencia)")
print(f"CV-AUC T_corr  = {cv_auc('T_corr'):.3f}")
print(f"CV-AUC HDX     = {cv_auc('HDX_corr'):.3f}")

## Calibratge

Escombrem $(k_1, k_2)$ i triem els que **maximitzen $|\rho|$** (separacio confort–temperatura).
El mapa de calor mostra el paisatge complet; el punt blanc és l'òptim.
Esperem físicament $k_2 > k_1 > 0$ (ple sol escalfa més que la barreja).

In [ ]:
# ----------------------- GRID SEARCH -----------------------
ks = np.arange(0, 20, 0.5)
H = np.zeros((len(ks), len(ks)))
best = (0.0, 0.0, -1.0)
for i, k1 in enumerate(ks):
    for j, k2 in enumerate(ks):
        rho = abs_rho(Trad(v, k1, k2), v.score)
        H[i, j] = rho
        if rho > best[2]:
            best = (k1, k2, rho)
K1, K2, RHO = best
print(f"optim:  k1(mix)={K1:.1f} C   k2(sol)={K2:.1f} C   |rho|={RHO:.3f}")
print(f"monotonia fisica k2>=k1: {K2>=K1}")
print(f"millora vs T_corr: +{RHO-abs_rho(v.T_corr,v.score):.3f}")

# desa la temperatura efectiva calibrada a nivell de vot i de parada
v['T_rad']  = Trad(v, K1, K2)
df['T_rad'] = Trad(df, K1, K2)

In [ ]:
# --- GRID SEARCH ACTUALITZAT ---
ks   = np.arange(0, 20.01, 0.5)
kw_g = np.arange(0, 20.01, 0.5)
best = (0.0, 0.0, -1.0)
H2   = np.zeros((len(ks), len(kw_g)))
for i, k2 in enumerate(ks):
    for j, kw in enumerate(kw_g):
        rho = abs_rho(v.T_corr + k2*v.f_sun - kw*v.wind_sc, v.score)
        H2[i, j] = rho
        if rho > best[2]: best = (k2, kw, rho)
K_SUN, K_WIND, RHO = best
print(f"k_sun={K_SUN:.1f}°C  k_wind={K_WIND:.1f}  |rho|={RHO:.3f}")
v['T_env'] = v.T_corr + K_SUN*v.f_sun - K_WIND*v.wind_sc
df['T_env'] = df.T_corr + K_SUN*df.f_sun - K_WIND*df.wind_sc

In [ ]:
# ----------------------- GRID SEARCH: MIX + SUN + WIND -----------------------

k_mix_grid  = np.arange(0, 15.01, 0.5)
k_sun_grid  = np.arange(0, 15.01, 0.5)
k_wind_grid = np.arange(0, 14.01, 0.5)

best = {
    'k_mix': 0.0,
    'k_sun': 0.0,
    'k_wind': 0.0,
    'rho': -np.inf
}

records = []

for k_mix in k_mix_grid:
    for k_sun in k_sun_grid:
        for k_wind in k_wind_grid:

            T_eff = (
                v['T_corr']
                + k_mix*v['f_mix']
                + k_sun*v['f_sun']
                - k_wind*v['wind_sc']
            )

            rho = abs_rho(T_eff, v['score'])

            records.append({
                'k_mix': k_mix,
                'k_sun': k_sun,
                'k_wind': k_wind,
                'rho': rho
            })

            if rho > best['rho']:
                best = {
                    'k_mix': k_mix,
                    'k_sun': k_sun,
                    'k_wind': k_wind,
                    'rho': rho
                }

grid_results = pd.DataFrame(records)

best

In [ ]:
K_MIX  = best['k_mix']
K_SUN  = best['k_sun']
K_WIND = best['k_wind']

v['T_env'] = (
    v['T_corr']
    + K_MIX*v['f_mix']
    + K_SUN*v['f_sun']
    - K_WIND*v['wind_sc']
)

df['T_env'] = (
    df['T_corr']
    + K_MIX*df['f_mix']
    + K_SUN*df['f_sun']
    - K_WIND*df['wind_sc']
)

print(f"k_mix  = {K_MIX:.2f} ºC")
print(f"k_sun  = {K_SUN:.2f} ºC")
print(f"k_wind = {K_WIND:.2f} ºC")
print(f"|rho|  = {best['rho']:.3f}")

In [ ]:
# ----------------------- HEATMAP sun and wind  -----------------------
fig, ax = plt.subplots(figsize=(5.4,4.4))
im = ax.imshow(H2, origin='lower', aspect='auto', cmap='viridis',
               extent=[ks[0],ks[-1],kw_g[0],kw_g[-1]])
ax.scatter([K_WIND],[K_SUN], c='white', edgecolor='k', s=80, zorder=3, label='òptim')
ax.axhline(K_SUN, color='white', lw=.5, ls=':'); ax.axvline(K_WIND, color='white', lw=.5, ls=':')
ax.set_xlabel('$k_2$  (offset vent, °C)'); ax.set_ylabel('$k_1$  (offset sol, °C)')
ax.set_title('Separació $|\\rho|$(T_rad, confort)')
cb = fig.colorbar(im, ax=ax); cb.set_label('|ρ| Spearman')
# banda de la literatura de radiacio per a sol ple (~10-15 C de "felt temperature")
ax.axvspan(10, 15, ymin=0, ymax=1, color='red', alpha=0.12)
ax.legend(loc='upper left'); plt.tight_layout(); plt.savefig('fig_heatmap.png'); plt.show()
print('Banda vermella = rang de la literatura de radiacio per a sol ple (~10-15 C).')
print(K_SUN, K_WIND)

## Comparació cap a cap

Tres mètriques complementàries per a `T_corr`, `HDX` i la `T_rad` calibrada.

In [ ]:
# ----------------------- HEAD-TO-HEAD -----------------------
tab = pd.DataFrame({
    'metric': ['|rho| (vot)','CV-AUC descomfort','Kruskal-Wallis H'],
    'T_corr':  [abs_rho(v.T_corr,v.score),  cv_auc('T_corr'),  kw_H('T_corr')],
    'HDX':     [abs_rho(v.HDX_corr,v.score), cv_auc('HDX_corr'),kw_H('HDX_corr')],
    'T_rad':   [abs_rho(v.T_rad,v.score),    cv_auc('T_rad'),   kw_H('T_rad')],
    'T_env':   [abs_rho(v.T_env,v.score),    cv_auc('T_env'),   kw_H('T_env')],
}).set_index('metric').round(3)
print(tab)

## Incertesa: bootstrap agrupat per caminada

Re-mostregem **caminades senceres** (no vots) amb reposició, re-optimitzem $(k_1,k_2)$ i mesurem la
millora $\Delta|\rho| = |\rho|_{T_\mathrm{rad}} - |\rho|_{T_\mathrm{corr}}$. Si l'IC 95% **exclou el zero**,
la millora global és robusta. (Graella més gruixuda dins del bootstrap per velocitat; apuja `N_BOOT` per a la versió final.)

In [ ]:
# ----------------------- CLUSTER BOOTSTRAP -----------------------
N_BOOT = 300
rng = np.random.default_rng(0)
walks = v['walk'].unique()
coarse = np.arange(0, 15.01, 1.5)
boot_d, boot_k1, boot_k2 = [], [], []
for _ in range(N_BOOT):
    samp = rng.choice(walks, size=len(walks), replace=True)
    vb = pd.concat([v[v['walk']==w] for w in samp], ignore_index=True)
    bb, bk = -1, (0,0)
    for k1 in coarse:
        for k2 in coarse:
            rho = abs_rho(Trad(vb,k1,k2), vb.score)
            if rho > bb: bb, bk = rho, (k1,k2)
    boot_d.append(bb - abs_rho(vb.T_corr, vb.score)); boot_k1.append(bk[0]); boot_k2.append(bk[1])
boot_d = np.array(boot_d)
lo, hi = np.percentile(boot_d,[2.5,97.5])
print(f"Δ|rho|  mitjana {boot_d.mean():.3f}   IC95% [{lo:.3f}, {hi:.3f}]")
print(f"fraccio de re-mostres amb millora>0: {(boot_d>0).mean():.2f}")
print(f"k2(sol) bootstrap: mediana {np.median(boot_k2):.1f}  IC95% [{np.percentile(boot_k2,2.5):.1f}, {np.percentile(boot_k2,97.5):.1f}]")

fig, ax = plt.subplots(figsize=(5,3.2))
ax.hist(boot_d, bins=30, color='#1D9E75', alpha=.85)
ax.axvline(0, color='k', lw=1); ax.axvline(boot_d.mean(), color='#D85A30', lw=2, label='mitjana')
ax.set_xlabel('Δ|ρ|  (T_rad − T_corr)'); ax.set_ylabel('re-mostres'); ax.legend()
plt.tight_layout(); plt.savefig('fig_bootstrap.png'); plt.show()

## Control de règim (el punt clau)

¿La millora és **dins** de caminada o només **entre** caminades?

- *Dins de caminada* → el sol seria una coordenada **local** de confort.
- *Només entre caminades* → el sol és una variable de **forçament/règim** (com l'HDX): posiciona les
  caminades a l'eix de descomfort però no resol el confort localment.

Comparem $|\rho|$ dins de cada caminada (test de Wilcoxon aparellat) i la correlació parcial controlant l'SVF.

In [ ]:
# ----------------------- WITHIN-WALK -----------------------
def within_rho(temp_vals):
    out = []
    tmp = v.assign(tt=temp_vals)
    for w, g in tmp.groupby('walk'):
        if g['score'].nunique() > 1 and len(g) >= 8:
            out.append(abs(stats.spearmanr(g['tt'], g['score']).statistic))
    return np.array(out)

wc = within_rho(v.T_corr)
wr = within_rho(v.T_rad)
we = within_rho(v.T_env)
W1 = stats.wilcoxon(wr, wc, zero_method='zsplit')
W2 = stats.wilcoxon(wr, we, zero_method='zsplit')
W3 = stats.wilcoxon(we, wc, zero_method='zsplit')
print(f"|rho| mitja DINS de caminada:  T_corr {np.nanmean(wc):.3f}   T_rad {np.nanmean(wr):.3f}   T_env {np.nanmean(we):.3f}   (n={len(wc)} caminades)")
print(f"Wilcoxon aparellat (T_rad vs T_corr dins caminada): p = {W1.pvalue:.3f}")
print(f"Wilcoxon aparellat (T_env vs T_rad dins caminada): p = {W2.pvalue:.3f}")
print(f"Wilcoxon aparellat (T_env vs T_corr dins caminada): p = {W3.pvalue:.3f}")

# correlacio parcial controlant SVF (residus de rang)
def partial_abs(a, b, ctrl):
    ra, rb, rc = stats.rankdata(a), stats.rankdata(b), stats.rankdata(ctrl)
    def resid(y, x):
        X = np.c_[np.ones_like(x), x]; coef = np.linalg.lstsq(X, y, rcond=None)[0]; return y - X@coef
    return abs(np.corrcoef(resid(ra, rc), resid(rb, rc))[0,1])
m = v.dropna(subset=['SVF_n'])
print(f"\nparcial |rho|(·, confort | SVF):  T_corr {partial_abs(m.T_corr,m.score,m.SVF_n):.3f}   T_rad {partial_abs(m.T_rad,m.score,m.SVF_n):.3f}   T_env {partial_abs(m.T_env,m.score,m.SVF_n):.3f}")

## P(descomfort) vs temperatura, amb el llindar dels 29 °C

Probabilitat empírica de descomfort per bins de temperatura, per a `T_corr` i `T_rad`,
amb el marcador del llindar d'estrès per calor (~29 °C) que apareix a la literatura de Barcelona.

In [ ]:
# ----------------------- P(disc) vs T -----------------------
def binned(col, nbins=9):
    q = pd.qcut(v[col], nbins, duplicates='drop')
    g = v.groupby(q)['disc']
    p = g.mean(); n = g.size(); ctr = v.groupby(q)[col].mean()
    # Wilson CI
    z=1.96; lo,hi=[],[]
    for pi,ni in zip(p,n):
        den=1+z*z/ni; cen=pi+z*z/(2*ni); half=z*np.sqrt(pi*(1-pi)/ni+z*z/(4*ni*ni))
        lo.append((cen-half)/den); hi.append((cen+half)/den)
    return ctr.values, p.values, np.array(lo), np.array(hi)

fig, ax = plt.subplots(figsize=(5.6,3.6))
for col, c, lab in [('T_env','#185FA5','T_env'), ('T_corr',"#0B1014",'T_corr'), ('T_rad','#D85A30','T_rad')]:
    x,p,ci_lo,ci_hi = binned(col)
    ax.plot(x, p, '-o', color=c, label=lab)
    ax.fill_between(x, ci_lo, ci_hi, color=c, alpha=.15)
ax.axvline(29, color='k', ls='--', lw=1); ax.text(29.1, ax.get_ylim()[1]*.92, '29 °C', fontsize=9)
ax.set_xlabel('temperatura (°C)'); ax.set_ylabel('P(descomfort)'); ax.legend()
plt.tight_layout(); plt.savefig('fig_pdisc.png'); plt.show()

In [ ]:
# ----------------------- SAVE OUTPUTS -----------------------
out_cols = ['space_code','space_name','city','walk','T_corr','HDX_corr','f_sun','f_mix','SVF_n','T_rad']
df[out_cols].to_csv('stops_with_Trad.csv', index=False)
summary = {
    'k1_mix': K1, 'k2_sun': K2, 
    'rho_Tcorr': abs_rho(v.T_corr,v.score), 'rho_HDX': abs_rho(v.HDX_corr,v.score), 'rho_Trad': RHO,
    'auc_Tcorr': cv_auc('T_corr'), 'auc_HDX': cv_auc('HDX_corr'), 'auc_Trad': cv_auc('T_rad'),
    'boot_dRho_mean': float(boot_d.mean()), 'boot_dRho_lo': float(ci_lo), 'boot_dRho_hi': float(ci_hi),
    'within_Tcorr': float(np.nanmean(wc)), 'within_Trad': float(np.nanmean(wr)), 'within_wilcoxon_p': float(W.pvalue),
}
pd.Series(summary).round(3).to_csv('Trad_summary.csv')
print('desat: stops_with_Trad.csv, Trad_summary.csv, fig_*.png')
pd.Series(summary).round(3)

## Com llegir els resultats (resum)

- **Sanity check físic.** Si $k_2$(sol) cau a ~10–15 °C, el calibratge cec a les dades coincideix amb la
  literatura de radiació → tens una història física, no un *fit* arbitrari.
- **Bootstrap.** Si l'IC de $\Delta|\rho|$ exclou el zero, la millora *global* és robusta.
- **Within-walk.** Si la millora **desapareix dins de caminada** (Wilcoxon no significatiu), el sol **no és
  una coordenada local** sinó un **forçament de règim** — separa caminades, no parades. Això **no** és un
  fracàs: és l'argument que reforça que T és la coordenada local i que sol/HDX són variables mesoscòpiques
  de forçament. Encaixa amb la teva narrativa i amb l'anàlisi de transicions sota forçament (pots *binnar*
  les transicions per `f_sun` igual que ho fas per HDX).
- **Recomanació de tesi.** Mantén `T_corr` com a coordenada local; presenta `T_rad`/sol com a **eix de
  forçament/règim**. Mitja pàgina + la figura de separació (T_corr vs HDX vs T_rad) i prou.
